# Level 3 — Business Analytics

## Objective

This notebook answers business questions using SQL queries and summarizes insights for stakeholders.

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT *
FROM read_csv_auto('../data/superstore_features.csv');
""")

con.execute("""
SELECT *
FROM superstore_features
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales,customer_span_days,customer_tier
0,7556,CA-2014-114181,2014-05-10,2014-05-14,Second Class,AF-10885,Art Foster,Consumer,United States,Philadelphia,Pennsylvania,19134,East,OFF-AR-10000716,Office Supplies,Art,DIXON Ticonderoga Erasable Checking Pencils,22.320,5,0.2,5.3010,4,0.237500,2014,5,861.565,915,Standard
1,2868,CA-2016-115588,2016-11-10,2016-11-12,Second Class,AF-10885,Art Foster,Consumer,United States,Toledo,Ohio,43615,East,OFF-SU-10001225,Office Supplies,Supplies,Staple remover,14.720,5,0.2,-3.3120,2,-0.225000,2016,11,861.565,915,Standard
2,3285,CA-2014-103702,2014-09-12,2014-09-17,Standard Class,AF-10885,Art Foster,Consumer,United States,Fairfield,Ohio,45014,East,OFF-BI-10002429,Office Supplies,Binders,"Premier Elliptical Ring Binder, Black",63.924,7,0.7,-46.8776,5,-0.733333,2014,9,861.565,915,Standard
3,2869,CA-2016-115588,2016-11-10,2016-11-12,Second Class,AF-10885,Art Foster,Consumer,United States,Toledo,Ohio,43615,East,OFF-ST-10001558,Office Supplies,Storage,Acco Perma 4000 Stacking Storage Drawers,38.976,3,0.2,-2.4360,2,-0.062500,2016,11,861.565,915,Standard
4,7555,CA-2014-114181,2014-05-10,2014-05-14,Second Class,AF-10885,Art Foster,Consumer,United States,Philadelphia,Pennsylvania,19134,East,FUR-BO-10004467,Furniture,Bookcases,Bestar Classic Bookcase,349.965,7,0.5,-216.9783,4,-0.620000,2014,5,861.565,915,Standard


## Selecting Variables for Business Analysis

The engineered dataset contains both the original retail variables and the features created in **Notebook 02**. While all variables remain available in the exported dataset, only those relevant to the business analyses in this notebook were selected.

The following columns were excluded for the reasons below:

- **`row_id`** was excluded because it serves only as a unique row identifier and does not provide meaningful information for the planned business analyses.

- **`order_date`** was excluded because the engineered features **`order_year`** and **`order_month`** provide the time granularity needed for this analysis.

- **`ship_date`** was excluded because the engineered feature **`fulfillment_days`** more directly measures shipping performance.

- **`customer_id`** was excluded in favor of **`customer_name`**, which produces more interpretable customer-level reports.

- **`product_id`** and **`product_name`** were excluded from product-level analysis because Notebook 01 identified inconsistencies in the relationship between these fields. Without sufficient information to determine the correct identifier-name relationships, using either field to make individual-product comparisons could produce unreliable conclusions. Product performance is therefore evaluated at the independently validated **`category`** and **`sub_category`** levels.

- **`country`** was excluded because every observation occurred in the United States, providing no additional analytical value.

- **`city`** and **`postal_code`** were excluded because they contain **531** and **631** unique values, respectively. For this analysis, **`region`** and **`state`** provide a more meaningful level of geographic aggregation while reducing unnecessary granularity.

The resulting dataset retains the variables most relevant to analyzing customer behavior, category and subcategory performance, geographic trends, profitability, and operational efficiency. This selection also ensures that the analyses rely only on fields whose relationships were validated during preprocessing, keeping the findings focused, interpretable, and defensible.



In [2]:
analysis_df = con.execute("""
SELECT
    -- Order
    order_id,

    -- Customer
    customer_name,
    segment,

    -- Geography
    region,
    state,

    -- Product
    category,
    sub_category,
    product_name,

    -- Shipping
    ship_mode,

    -- Original Sales Metrics
    sales,
    quantity,
    discount,
    profit,

    -- Engineered Features
    order_year,
    order_month,
    fulfillment_days,
    profit_margin,
    customer_lifetime_sales,
    customer_tier

FROM superstore_features
""").df()

analysis_df.head().style.hide(axis="index")

order_id,customer_name,segment,region,state,category,sub_category,product_name,ship_mode,sales,quantity,discount,profit,order_year,order_month,fulfillment_days,profit_margin,customer_lifetime_sales,customer_tier
CA-2014-114181,Art Foster,Consumer,East,Pennsylvania,Office Supplies,Art,DIXON Ticonderoga Erasable Checking Pencils,Second Class,22.320000,5,0.200000,5.301000,2014,5,4,0.237500,861.565000,Standard
CA-2016-115588,Art Foster,Consumer,East,Ohio,Office Supplies,Supplies,Staple remover,Second Class,14.720000,5,0.200000,-3.312000,2016,11,2,-0.225000,861.565000,Standard
CA-2014-103702,Art Foster,Consumer,East,Ohio,Office Supplies,Binders,"Premier Elliptical Ring Binder, Black",Standard Class,63.924000,7,0.700000,-46.877600,2014,9,5,-0.733333,861.565000,Standard
CA-2016-115588,Art Foster,Consumer,East,Ohio,Office Supplies,Storage,Acco Perma 4000 Stacking Storage Drawers,Second Class,38.976000,3,0.200000,-2.436000,2016,11,2,-0.062500,861.565000,Standard
CA-2014-114181,Art Foster,Consumer,East,Pennsylvania,Furniture,Bookcases,Bestar Classic Bookcase,Second Class,349.965000,7,0.500000,-216.978300,2014,5,4,-0.620000,861.565000,Standard


## Customer Analysis

Customer analytics focuses on understanding purchasing behavior across individual customers and customer segments. By identifying high-value customers, comparing segment performance, and evaluating customer lifetime sales, we can better understand who generates the greatest value for the business and where customer retention efforts should be focused.

### Top Customers by Lifetime Sales

**Question:**

Who are the ten highest-value customers based on lifetime sales, and what do their ordering frequency, average spending per order, and observed customer span reveal about their purchasing behavior?

**Objective:**

Identify the ten customers with the highest cumulative sales and compare their total number of orders, average spending per order, and the number of days between their first and most recent recorded purchases. This analysis helps determine whether high customer value is driven primarily by frequent purchasing, larger orders, a longer purchasing history, or a combination of these factors.

**Variables:**

- `customer_name`
- `order_id`
- `order_date`
- `sales`
- `customer_lifetime_sales`
- `customer_span_days`

In [3]:
con.execute("""
SELECT
    customer_name,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS customer_lifetime_sales,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_spend_per_order,
    ANY_VALUE(customer_span_days) AS customer_span_days

FROM superstore_features

GROUP BY customer_id, customer_name

ORDER BY customer_lifetime_sales DESC

LIMIT 10;
""").df().style.hide(axis="index").format({
    "customer_lifetime_sales": "${:,.2f}",
    "total_orders": "{:,}",
    "avg_spend_per_order": "${:,.2f}",
    "customer_span_days": "{:,} days"
})

customer_name,total_orders,customer_lifetime_sales,avg_spend_per_order,customer_span_days
Sean Miller,5,"$25,043.05","$5,008.61","1,304 days"
Tamara Chand,5,"$19,052.22","$3,810.44",750 days
Raymond Buch,6,"$15,117.34","$2,519.56",542 days
Tom Ashbrook,4,"$14,595.62","$3,648.91","1,136 days"
Adrian Barton,10,"$14,473.57","$1,447.36","1,065 days"
Ken Lonsdale,12,"$14,175.23","$1,181.27","1,207 days"
Sanjit Chand,9,"$14,142.33","$1,571.37","1,068 days"
Hunter Lopez,6,"$12,873.30","$2,145.55","1,397 days"
Sanjit Engle,11,"$12,209.44","$1,109.95","1,350 days"
Christopher Conant,5,"$12,129.07","$2,425.81",543 days


**Results:**

Sean Miller was the highest-value customer, generating **$25,043.05** in lifetime sales. Tamara Chand ranked second with **$19,052.22**, followed by Raymond Buch with **$15,117.34**. Together, the ten highest-value customers generated **$153,811.17** in lifetime sales.

**Key Insights:**

The results identify a small group of customers with substantial purchasing histories. Sean Miller generated approximately **$5,991 more** than the second-ranked customer, making him a particularly important account. These customers may be strong candidates for personalized promotions, loyalty rewards, and targeted retention efforts.

### Sales Performance by Customer Segment

**Question:**
Which customer segments generate the most revenue?

**Objective:**

Compare sales performance across the Consumer, Corporate, and Home Office segments to determine which customer groups contribute the greatest share of revenue. This analysis helps identify where the business generates the majority of its sales and whether certain segments warrant additional marketing or sales efforts.

**Variables:**
- `segment`
- `sales`
- `profit`
- `cusomer_id`

In [4]:
con.execute("""
SELECT
    segment,

    SUM(sales) AS total_sales,

    SUM(profit) AS total_profit,

    100.0 * SUM(sales) / SUM(SUM(sales)) OVER ()
        AS percentage_of_total_sales,

    100.0 * SUM(profit) / SUM(sales)
        AS profit_margin_percentage,

    COUNT(DISTINCT customer_id) AS customer_count,

    SUM(sales) / COUNT(DISTINCT customer_id)
        AS avg_sales_per_customer,


FROM superstore_features
GROUP BY segment
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "customer_count": "{:,}",
    "total_sales": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%",
    "avg_sales_per_customer": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

segment,total_sales,total_profit,percentage_of_total_sales,profit_margin_percentage,customer_count,avg_sales_per_customer
Consumer,"$1,161,401.34","$134,119.21",50.56%,11.55%,409,"$2,839.61"
Corporate,"$706,146.37","$91,979.13",30.74%,13.03%,236,"$2,992.15"
Home Office,"$429,653.15","$60,298.68",18.70%,14.03%,148,"$2,903.06"


**Results:**

The Consumer segment generated the highest sales at **$1,161,401.34**, representing **50.56%** of total revenue. Corporate customers contributed **$706,146.37**, or **30.74%**, while the Home Office segment generated **$429,653.15**, accounting for the remaining **18.70%**.

**Key Insights:**

Consumer customers are the company’s largest source of revenue, producing slightly more than half of all sales. However, the Corporate segment also makes a substantial contribution, with Consumer and Corporate customers together accounting for **81.30%** of total sales. Although Home Office is the smallest segment, additional profitability analysis is needed before concluding that it is less valuable, since revenue alone does not account for costs or profit margins.

### Sales and Profitability by Customer Tier

**Question:**
How do sales and profitability differ across customer tiers?

**Objective:**
Compare total sales, total profit, and profit margins across customer tiers to determine whether higher-value customers also generate stronger profitability. This analysis helps distinguish customers who produce substantial revenue from those who create the greatest financial value for the business.

**Variables:**
`customer_tier`
`sales`
`profit`
`profit_margin`


In [5]:
con.execute("""
SELECT
    customer_tier,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM superstore_features
GROUP BY customer_tier
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})


customer_tier,total_sales,total_profit,profit_margin_percentage
Big Fish,"$709,342.71","$122,167.79",17.22%
High Value,"$574,885.76","$52,925.88",9.21%
Premium,"$557,107.99","$65,687.41",11.79%
Standard,"$455,864.40","$45,615.94",10.01%


**Results:**

Big Fish customers generated the highest total sales at **$709,342.71** and the highest total profit at **$122,167.79**. They also achieved the strongest profit margin at **17.22%**.

High Value customers generated slightly more sales than Premium customers—**$574,885.76** compared with **$557,107.99**—but Premium customers produced considerably more profit: **$65,687.41** compared with **$52,925.88**. Premium customers also had a higher profit margin of **11.79%**, while the High Value tier had the lowest margin at **9.21%**.

**Key Insights:**

Customer value based on lifetime sales does not correspond perfectly with profitability. Although High Value customers generated more revenue than Premium customers, they produced approximately **$12,761 less profit** and had a profit margin that was **2.58 percentage points lower**.

Big Fish customers are especially valuable because they lead the other tiers in sales, total profit, and profit margin. The relatively weak margin among High Value customers warrants further investigation into factors such as discount usage, product mix, and shipping costs. Retention strategies should therefore consider profitability in addition to lifetime sales when identifying the company’s most valuable customers.

## Geographic Analysis

Geographic analysis evaluates business performance across regions and states. Comparing sales, profitability, and customer activity by location helps identify high-performing markets, uncover regional trends, and highlight areas that may benefit from targeted business strategies.

## Product Analysis

Product analysis examines sales and profitability across product categories, subcategories, and individual products. The objective is to identify top-performing products, recognize underperforming inventory, and understand which areas of the product portfolio contribute most to overall business success.

## Sales & Profitability Analysis

Sales and profitability analysis investigates the financial performance of the business by examining revenue, discounts, profit, and profit margins. This section explores how pricing and discounting strategies influence profitability and identifies opportunities to improve financial performance.

## Shipping & Time Analysis

Shipping and time analysis evaluates operational efficiency and temporal business trends. By analyzing fulfillment times, shipping methods, and sales performance across months and years, this section identifies seasonal patterns, monitors delivery performance, and uncovers trends that can support operational planning.

In [6]:
con.sql("""
SELECT
    order_year,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    SUM(sales) / COUNT(DISTINCT order_id) AS avg_order_value,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
""").df().style.hide(axis="index").format({
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "avg_order_value": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})

order_year,total_orders,total_sales,total_profit,avg_order_value,profit_margin_percentage
2014,969,"$484,247.50","$49,543.97",$499.74,10.23%
2015,1038,"$470,532.51","$61,618.60",$453.31,13.10%
2016,1315,"$609,205.60","$81,795.17",$463.27,13.43%
2017,1687,"$733,215.26","$93,439.27",$434.63,12.74%


### Do Product Prices Differ Across Customer Segments?

Average unit prices and discounts were compared across customer segments to determine whether pricing patterns differed between Consumer, Corporate, and Home Office customers.

In [7]:
con.execute('''
SELECT 
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore_features
GROUP BY product_id, segment
ORDER BY product_id
Limit 40''').df()           

,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Home Office,102.83,0.150
2,FUR-BO-10000330,Consumer,111.91,0.075
3,FUR-BO-10000362,Home Office,145.33,0.150
4,FUR-BO-10000362,Corporate,158.16,0.075
5,FUR-BO-10000362,Consumer,136.78,0.200
6,FUR-BO-10000468,Corporate,48.58,0.000
7,FUR-BO-10000468,Consumer,37.89,0.220
8,FUR-BO-10000711,Consumer,70.98,0.000
9,FUR-BO-10000711,Home Office,70.98,0.000


In [8]:
con.execute ("""
SELECT 
    segment,
    COUNT(DISTINCT customer_name) AS customer_count,
    ROUND(SUM(sales)) AS total_sales
FROM superstore_features
GROUP BY segment
""").df()


,segment,customer_count,total_sales
0,Home Office,148,429653.0
1,Consumer,409,1161401.0
2,Corporate,236,706146.0


In [9]:
con.execute('''
SELECT
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore_features
GROUP BY product_id, segment
ORDER BY product_id
LIMIT 40;
''').df()

,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Consumer,111.91,0.075
2,FUR-BO-10000330,Home Office,102.83,0.150
3,FUR-BO-10000362,Home Office,145.33,0.150
4,FUR-BO-10000362,Consumer,136.78,0.200
5,FUR-BO-10000362,Corporate,158.16,0.075
6,FUR-BO-10000468,Consumer,37.89,0.220
7,FUR-BO-10000468,Corporate,48.58,0.000
8,FUR-BO-10000711,Consumer,70.98,0.000
9,FUR-BO-10000711,Home Office,70.98,0.000


In [10]:
con.execute('''
SELECT
    segment,
    ROUND(AVG(fulfillment_days),2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MEDIAN(fulfillment_days) AS median_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY segment
ORDER BY avg_fulfillment_days;
''').df()

,segment,avg_fulfillment_days,min_days,median_days,max_days
0,Home Office,3.92,0,4.0,7
1,Consumer,3.94,0,4.0,7
2,Corporate,4.01,0,4.0,7


In [11]:
con.execute("""
SELECT
    ship_mode,
    ROUND(AVG(fulfillment_days), 2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY ship_mode
ORDER BY avg_fulfillment_days;
""").df()

,ship_mode,avg_fulfillment_days,min_days,max_days
0,Same Day,0.04,0,1
1,First Class,2.18,1,4
2,Second Class,3.24,1,5
3,Standard Class,5.01,3,7


extra stuff to think about: 

In [12]:
con.execute(''' 
SELECT
    product_name,
    sales,
    profit,
    profit_margin
FROM superstore_features
ORDER BY profit_margin
LIMIT 10
''').df() 

,product_name,sales,profit,profit_margin
0,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.624,-4.4660,-2.75
1,Hoover Shoulder Vac Commercial Portable Vacuum,143.128,-393.6020,-2.75
2,Kensington 6 Outlet SmartSocket Surge Protector,24.588,-67.6170,-2.75
3,Hoover Portapower Portable Vacuum,2.688,-7.3920,-2.75
4,Fellowes 8 Outlet Superior Workstation Surge P...,33.620,-90.7740,-2.70
5,Euro Pro Shark Stick Mini Vacuum,48.784,-131.7168,-2.70
6,Acco Smartsocket Color-Coded Six-Outlet AC Ada...,26.406,-71.2962,-2.70
7,Hoover Commercial Lightweight Upright Vacuum w...,93.032,-251.1864,-2.70
8,Hoover Commercial Lightweight Upright Vacuum,1.392,-3.7584,-2.70
9,Belkin 6 Outlet Metallic Surge Strip,4.356,-11.7612,-2.70
